# Blood Pressure Analysis
Predicting systolic and diastolic BP using OLS regression.

## 1. Imports & Settings

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

DATA_PATH   = '../data/processed/final_dataset_de.csv'
FIGURES_DIR = '../presentation/figures'
os.makedirs(FIGURES_DIR, exist_ok=True)

# colors
SYS_C = '#c0392b'  # red for systolic
DIA_C = '#2980b9'  # blue for diastolic
GRAY  = '#bdc3c7'
BG    = '#f8f9fa'

features = ['age', 'gender_male', 'raucher_int', 'blutzucker_int',
            'cholesterin_int', 'in_behandlung_int', 'befinden']

labels = ['Age', 'Male Sex', 'Smoker', 'Known Blood Sugar',
          'Known Cholesterol', 'In Treatment', 'Wellbeing']

## 2. Load Data & Feature Engineering

In [2]:
df = pd.read_csv(DATA_PATH)

# convert categorical columns to binary integers
df['gender_male']       = (df['geschlecht'] == 'm').astype(int)
df['raucher_int']       = df['raucher'].astype(int)
df['blutzucker_int']    = df['blutzucker_bekannt'].astype(int)
df['cholesterin_int']   = df['cholesterin_bekannt'].astype(int)
df['in_behandlung_int'] = df['in_behandlung'].astype(int)

df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 30, 40, 50, 60, 70, 200],
    labels=['<30', '30-39', '40-49', '50-59', '60-69', '70+']
)

# keep only rows with no missing values
mdf = df[features + ['messwert_bp_sys', 'messwert_bp_dia']].dropna()

print(f"Patients:  {len(mdf):,}")
print(f"Systolic:  {mdf['messwert_bp_sys'].mean():.1f} +/- {mdf['messwert_bp_sys'].std():.1f} mmHg")
print(f"Diastolic: {mdf['messwert_bp_dia'].mean():.1f} +/- {mdf['messwert_bp_dia'].std():.1f} mmHg")

Patients:  15,219
Systolic:  125.1 +/- 19.3 mmHg
Diastolic: 82.6 +/- 14.3 mmHg


## 3. OLS Regression Models

In [3]:
X = sm.add_constant(mdf[features])

model_sys = sm.OLS(mdf['messwert_bp_sys'], X).fit()
model_dia = sm.OLS(mdf['messwert_bp_dia'], X).fit()

print("=== Systolic BP ===")
print(model_sys.summary())

=== Systolic BP ===
                            OLS Regression Results                            
Dep. Variable:        messwert_bp_sys   R-squared:                       0.155
Model:                            OLS   Adj. R-squared:                  0.154
Method:                 Least Squares   F-statistic:                     397.4
Date:                Fri, 29 May 2026   Prob (F-statistic):               0.00
Time:                        11:25:35   Log-Likelihood:                -65379.
No. Observations:               15219   AIC:                         1.308e+05
Df Residuals:                   15211   BIC:                         1.308e+05
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
const             

In [4]:
print("=== Diastolic BP ===")
print(model_dia.summary())

=== Diastolic BP ===
                            OLS Regression Results                            
Dep. Variable:        messwert_bp_dia   R-squared:                       0.040
Model:                            OLS   Adj. R-squared:                  0.039
Method:                 Least Squares   F-statistic:                     89.57
Date:                Fri, 29 May 2026   Prob (F-statistic):          1.80e-128
Time:                        11:25:35   Log-Likelihood:                -61799.
No. Observations:               15219   AIC:                         1.236e+05
Df Residuals:                   15211   BIC:                         1.237e+05
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
const            

In [5]:
# extract coefficients, confidence intervals, p-values (skip intercept row)
coef_s = model_sys.params[1:].values
lo_s   = model_sys.conf_int().iloc[1:, 0].values
hi_s   = model_sys.conf_int().iloc[1:, 1].values
pv_s   = model_sys.pvalues[1:].values

coef_d = model_dia.params[1:].values
lo_d   = model_dia.conf_int().iloc[1:, 0].values
hi_d   = model_dia.conf_int().iloc[1:, 1].values
pv_d   = model_dia.pvalues[1:].values

## 4. Cross-Validation (5-Fold R²)

In [6]:
X_scaled = StandardScaler().fit_transform(mdf[features].values)

cv_sys = cross_val_score(LinearRegression(), X_scaled, mdf['messwert_bp_sys'].values, cv=5, scoring='r2')
cv_dia = cross_val_score(LinearRegression(), X_scaled, mdf['messwert_bp_dia'].values, cv=5, scoring='r2')

print(f"Systolic  CV R²: {cv_sys.mean():.4f} +/- {cv_sys.std():.4f}")
print(f"Diastolic CV R²: {cv_dia.mean():.4f} +/- {cv_dia.std():.4f}")

Systolic  CV R²: 0.1300 +/- 0.0309
Diastolic CV R²: 0.0217 +/- 0.0110


## 5. Group Comparisons

In [7]:
def sig(p):
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'

def compare_groups(label, mask):
    for bp, col in [('Systolic', 'messwert_bp_sys'), ('Diastolic', 'messwert_bp_dia')]:
        g1 = df.loc[mask, col]
        g2 = df.loc[~mask, col]
        _, p = stats.ttest_ind(g1, g2)
        print(f"  {label} | {bp}: {g1.mean():.1f} vs {g2.mean():.1f}  diff={g1.mean()-g2.mean():+.1f} mmHg  {sig(p)}")

compare_groups('Male vs Female',           df['geschlecht'] == 'm')
compare_groups('Smoker vs Non-Smoker',     df['raucher'] == True)
compare_groups('Blood Sugar Known vs Not', df['blutzucker_bekannt'] == True)
compare_groups('Cholesterol Known vs Not', df['cholesterin_bekannt'] == True)
compare_groups('In Treatment vs Not',      df['in_behandlung'] == True)

r_sys, _ = stats.pearsonr(mdf['age'], mdf['messwert_bp_sys'])
r_dia, _ = stats.pearsonr(mdf['age'], mdf['messwert_bp_dia'])
print(f"\nAge ~ Systolic:  r={r_sys:.3f}, +{coef_s[0]:.2f} mmHg/year")
print(f"Age ~ Diastolic: r={r_dia:.3f}, +{coef_d[0]:.2f} mmHg/year")

  Male vs Female | Systolic: 128.0 vs 122.8  diff=+5.1 mmHg  ***
  Male vs Female | Diastolic: 84.6 vs 81.0  diff=+3.7 mmHg  ***
  Smoker vs Non-Smoker | Systolic: 122.9 vs 125.5  diff=-2.6 mmHg  ***
  Smoker vs Non-Smoker | Diastolic: 82.5 vs 82.6  diff=-0.1 mmHg  ns
  Blood Sugar Known vs Not | Systolic: 128.8 vs 123.7  diff=+5.1 mmHg  ***
  Blood Sugar Known vs Not | Diastolic: 83.8 vs 82.1  diff=+1.7 mmHg  ***
  Cholesterol Known vs Not | Systolic: 128.0 vs 123.4  diff=+4.6 mmHg  ***
  Cholesterol Known vs Not | Diastolic: 83.5 vs 82.0  diff=+1.6 mmHg  ***
  In Treatment vs Not | Systolic: 137.8 vs 123.2  diff=+14.6 mmHg  ***
  In Treatment vs Not | Diastolic: 86.4 vs 82.0  diff=+4.4 mmHg  ***

Age ~ Systolic:  r=0.347, +0.32 mmHg/year
Age ~ Diastolic: r=0.144, +0.10 mmHg/year


## 6. Helper Functions

In [8]:
def stars(p):
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''

def fmt_p(p):
    if p < 0.001: return '<0.001***'
    elif p < 0.01: return f'{p:.3f}**'
    elif p < 0.05: return f'{p:.3f}*'
    else: return f'{p:.3f} ns'

def save_fig(fig, filename):
    path = f'{FIGURES_DIR}/{filename}'
    fig.savefig(path, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f'Saved: {path}')

def style_table(table):
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    for (r, c), cell in table.get_celld().items():
        cell.set_edgecolor('#dfe6e9')
        if r == 0:
            cell.set_facecolor('#2c3e50')
            cell.set_text_props(color='white', fontweight='bold')
        elif r % 2 == 0:
            cell.set_facecolor('#eaf4fb')
        else:
            cell.set_facecolor('white')
        if r > 0 and c in [2, 4]:
            t = cell.get_text().get_text()
            if '***' in t: cell.set_facecolor('#fadbd8')
            elif '*' in t: cell.set_facecolor('#fdebd0')

## 7. Figure 1 — Forest Plot

In [9]:
fig, ax = plt.subplots(figsize=(11, 6))
fig.patch.set_facecolor(BG)
ax.set_facecolor('white')

for i, (c, l, h, p) in enumerate(zip(coef_s, lo_s, hi_s, pv_s)):
    color = SYS_C if p < 0.05 else GRAY
    ax.scatter(c, i + 0.2, color=color, s=100, zorder=5)
    ax.plot([l, h], [i + 0.2, i + 0.2], color=color, lw=2.2, zorder=4)
    if stars(p):
        ax.text(max(h, c) + 0.1, i + 0.2, stars(p), va='center', fontsize=10,
                color='#7f1d1d', fontweight='bold')

for i, (c, l, h, p) in enumerate(zip(coef_d, lo_d, hi_d, pv_d)):
    color = DIA_C if p < 0.05 else GRAY
    ax.scatter(c, i - 0.2, color=color, s=100, zorder=5)
    ax.plot([l, h], [i - 0.2, i - 0.2], color=color, lw=2.2, zorder=4)
    if stars(p):
        ax.text(max(h, c) + 0.1, i - 0.2, stars(p), va='center', fontsize=10,
                color='#1a5276', fontweight='bold')

ax.axvline(0, color='black', lw=1.2, linestyle='--', alpha=0.6)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=12)
ax.set_xlabel('Regression Coefficient β (mmHg)', fontsize=12)
ax.set_title(
    f'Figure 1: Regression Coefficients (95% CI)\n'
    f'Systolic Adj-R²={model_sys.rsquared_adj:.3f}  |  Diastolic Adj-R²={model_dia.rsquared_adj:.3f}',
    fontsize=11, fontweight='bold'
)
ax.legend(handles=[
    Patch(color=SYS_C, label=f'Systolic (CV R²={cv_sys.mean():.3f})'),
    Patch(color=DIA_C, label=f'Diastolic (CV R²={cv_dia.mean():.3f})')
], fontsize=11, loc='lower right')
ax.grid(axis='x', alpha=0.3)

save_fig(fig, 'fig1_forest_plot.png')

Saved: ../presentation/figures/fig1_forest_plot.png


## 8. Figure 2 — Coefficient Table

In [10]:
table_rows = [
    [lbl, f'{coef_s[i]:+.3f}', fmt_p(pv_s[i]), f'{coef_d[i]:+.3f}', fmt_p(pv_d[i])]
    for i, lbl in enumerate(labels)
]

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor(BG)
ax.axis('off')

table = ax.table(
    cellText=table_rows,
    colLabels=['Predictor', 'β Systolic', 'p-value', 'β Diastolic', 'p-value'],
    cellLoc='center',
    loc='center',
    bbox=[0, 0, 1, 1]
)
style_table(table)
ax.set_title(
    'Figure 2: Coefficient Summary\n(β = mmHg per unit  |  ***p<0.001  **p<0.01  *p<0.05)',
    fontsize=11, fontweight='bold', pad=15
)

save_fig(fig, 'fig2_coefficient_table.png')

Saved: ../presentation/figures/fig2_coefficient_table.png


## 9. Figure 3 — Mean BP by Age Group

In [11]:
age_labels = ['<30', '30-39', '40-49', '50-59', '60-69', '70+']
ag = df.groupby('age_group', observed=True)[['messwert_bp_sys', 'messwert_bp_dia']].mean()

fig, ax = plt.subplots(figsize=(9, 6))
fig.patch.set_facecolor(BG)
ax.set_facecolor('white')

x = np.arange(len(age_labels))
ax.plot(x, ag['messwert_bp_sys'].values, 'o-', color=SYS_C, lw=2.5, ms=9, label='Systolic BP')
ax.plot(x, ag['messwert_bp_dia'].values, 's-', color=DIA_C, lw=2.5, ms=9, label='Diastolic BP')

ax.axhline(140, color=SYS_C, lw=1.2, linestyle=':', alpha=0.6)
ax.text(5.05, 141, 'HT\n(>=140)', fontsize=8.5, color=SYS_C)
ax.axhline(90, color=DIA_C, lw=1.2, linestyle=':', alpha=0.6)
ax.text(5.05, 91, 'HT\n(>=90)', fontsize=8.5, color=DIA_C)

for xi, s, d in zip(x, ag['messwert_bp_sys'].values, ag['messwert_bp_dia'].values):
    ax.text(xi, s + 0.9, f'{s:.1f}', ha='center', fontsize=9, color=SYS_C, fontweight='bold')
    ax.text(xi, d - 2.8, f'{d:.1f}', ha='center', fontsize=9, color=DIA_C, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(age_labels, fontsize=11)
ax.set_ylabel('Mean BP (mmHg)', fontsize=12)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
ax.set_title('Figure 3: Mean Blood Pressure by Age Group', fontsize=11, fontweight='bold')

save_fig(fig, 'fig3_bp_by_age_group.png')

Saved: ../presentation/figures/fig3_bp_by_age_group.png


## 10. Figure 4 — Systolic BP by Clinical Groups

In [12]:
group_names = ['Female', 'Male', 'Non-\nSmoker', 'Smoker',
               'No Blood\nSugar', 'Blood\nSugar', 'Not In\nTreatment', 'In\nTreatment']

sys_means = [
    df[df['geschlecht'] == 'f']['messwert_bp_sys'].mean(),
    df[df['geschlecht'] == 'm']['messwert_bp_sys'].mean(),
    df[df['raucher'] == False]['messwert_bp_sys'].mean(),
    df[df['raucher'] == True]['messwert_bp_sys'].mean(),
    df[df['blutzucker_bekannt'] == False]['messwert_bp_sys'].mean(),
    df[df['blutzucker_bekannt'] == True]['messwert_bp_sys'].mean(),
    df[df['in_behandlung'] == False]['messwert_bp_sys'].mean(),
    df[df['in_behandlung'] == True]['messwert_bp_sys'].mean(),
]
bar_colors = ['#f1948a', '#c0392b', '#a9cce3', '#2980b9',
              '#a9dfbf', '#1e8449', '#f0b27a', '#d35400']

fig, ax = plt.subplots(figsize=(11, 6))
fig.patch.set_facecolor(BG)
ax.set_facecolor('white')

bars = ax.bar(range(8), sys_means, color=bar_colors, edgecolor='white', width=0.65)
for bar, val in zip(bars, sys_means):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            f'{val:.1f}', ha='center', fontsize=10, fontweight='bold')

ax.set_xticks(range(8))
ax.set_xticklabels(group_names, fontsize=10)
ax.set_ylabel('Mean Systolic BP (mmHg)', fontsize=12)
ax.set_ylim(100, 148)
ax.axhline(140, color='red', lw=1.2, linestyle=':', alpha=0.5)
ax.text(7.55, 140.5, 'HT\n(>=140)', fontsize=8, color='red')
for xv in [1.5, 3.5, 5.5]:
    ax.axvline(xv, color='gray', lw=0.8, linestyle='--', alpha=0.4)
ax.set_title('Figure 4: Mean Systolic BP by Clinical Groups', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

save_fig(fig, 'fig4_bp_by_groups.png')

Saved: ../presentation/figures/fig4_bp_by_groups.png


## 11. Figure 5 — Age vs BP Scatter

In [13]:
sample = mdf.sample(2000, random_state=42)
slope_s, int_s, *_ = stats.linregress(mdf['age'], mdf['messwert_bp_sys'])
slope_d, int_d, *_ = stats.linregress(mdf['age'], mdf['messwert_bp_dia'])
x_range = np.linspace(13, 95, 100)

fig, ax = plt.subplots(figsize=(9, 6))
fig.patch.set_facecolor(BG)
ax.set_facecolor('white')

ax.scatter(sample['age'], sample['messwert_bp_sys'], alpha=0.15, color=SYS_C, s=12)
ax.scatter(sample['age'], sample['messwert_bp_dia'], alpha=0.15, color=DIA_C, s=12)
ax.plot(x_range, slope_s * x_range + int_s, color=SYS_C, lw=2.8,
        label=f'Systolic  (r={r_sys:.2f}, p<0.001)')
ax.plot(x_range, slope_d * x_range + int_d, color=DIA_C, lw=2.8,
        label=f'Diastolic (r={r_dia:.2f}, p<0.001)')

ax.set_xlabel('Age (years)', fontsize=12)
ax.set_ylabel('BP (mmHg)', fontsize=12)
ax.set_title('Figure 5: Age vs Blood Pressure (n=2,000 sample)', fontsize=11, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

save_fig(fig, 'fig5_age_vs_bp_scatter.png')

Saved: ../presentation/figures/fig5_age_vs_bp_scatter.png
